Trimming is a technique used to address the problem of context overflow in Large Language Models (LLMs). During long conversations, the total number of tokens may exceed the model’s context window limit. To prevent this, trimming is applied.

In trimming, the system continuously checks whether the conversation history exceeds the allowed context window. If it does, only the most recent *N* messages are sent to the LLM, ensuring that the input remains within the token limit.

However, this approach has a trade-off: the model may lose awareness of older parts of the conversation because those earlier messages are no longer included in the prompt sent to the LLM.

It is important to note that trimming does not delete previous messages from the application state or memory. The older messages are still stored internally; they are simply excluded from the context provided to the LLM during inference.


In [10]:
from langgraph.graph import StateGraph , START , END , MessagesState
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages.utils import trim_messages , count_tokens_approximately

In [11]:
load_dotenv()
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash-lite')

In [12]:
MAX_TOKENS = 50

In [13]:
def call_model(state: MessagesState):
    
    # Trim conversation history -> last N messages that fit within the token budget
    messages = trim_messages(
        state["messages"],
        strategy="last",                      
        token_counter=count_tokens_approximately,
        max_tokens=MAX_TOKENS
    )

    print('Current Token Count ->', count_tokens_approximately(messages=messages))

    for message in messages:
        print(message.content)

    response = llm.invoke(messages)

    return {"messages": [response]}

In [14]:
graph = StateGraph(MessagesState)
graph.add_node('call_model' , call_model)

graph.add_edge(START , 'call_model')
graph.add_edge('call_model' , END)

checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)

In [15]:
graph

In [16]:
config = {"configurable": {"thread_id": "chat-1"}}

result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Imran."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 10
Hi, my name is Imran.


"Hi Imran! It's nice to meet you. How can I help you today?"

In [17]:
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "I am learning LangGraph."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 40
Hi, my name is Imran.
Hi Imran! It's nice to meet you. How can I help you today?
I am learning LangGraph.


"That's fantastic, Imran! LangGraph is a really powerful and exciting library for building complex, stateful applications with LLMs.\n\nWhat brings you to LangGraph? Are you working on a specific project, or are you just exploring its capabilities?\n\nI'd be happy to help you with your learning journey. We can discuss:\n\n*   **Core concepts of LangGraph:** Nodes, edges, states, workflows, etc.\n*   **Specific examples or use cases:** Agentic workflows, multi-step reasoning, data processing, etc.\n*   **Troubleshooting or debugging:** If you're running into any issues.\n*   **Best practices and tips.**\n*   **Anything else you're curious about!**\n\nJust let me know what's on your mind, and I'll do my best to assist you. What's the first thing you'd like to dive into?"

In [18]:
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "Can you explain short term memory?"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 13
Can you explain short term memory?


'Short-term memory (STM) is a fascinating and crucial aspect of our cognitive abilities. It\'s like a temporary holding space for information that we are actively using or thinking about right now. Think of it as the mental scratchpad or RAM of your brain.\n\nHere\'s a breakdown of its key characteristics and functions:\n\n**What is Short-Term Memory?**\n\n*   **Limited Capacity:** STM can only hold a small amount of information at any given time. The classic estimate is around **7 plus or minus 2 items** (George Miller\'s "magical number seven"). This means you can typically remember about 5 to 9 pieces of information.\n*   **Limited Duration:** Information in STM doesn\'t stay there for long. Without active effort to maintain it, it fades away relatively quickly, usually within **15 to 30 seconds**.\n*   **Active and Conscious:** STM is where our conscious awareness resides. The information you are currently thinking about, processing, or manipulating is in your short-term memory.\n*

In [ ]:
result = workflow.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config,
)

result["messages"][-1].content

### It will not be able to answer my name because of trimming its lost the context of most previous messages

Current Token Count -> 8
What is my name?


'I do not know your name. As an AI, I do not have access to any personal information about you, including your name.'